# Module 06: Interactive Error Handling & Structured Logging

### What You Will Discover
By running this notebook, you will explore exception hierarchies, explicit exception chaining (`raise from`), exception suppression traps, and structured JSON logging.

**Key Question Answered:** *Why does `except Exception:` catch fatal errors you never intended to handle, and how does explicit exception chaining preserve original root causes?*


In [ ]:
# Step 1: Base Custom Domain Exception Hierarchy
class ApplicationError(Exception):
    """Base exception for all domain errors."""

class DatabaseError(ApplicationError):
    """Raised when database operations fail."""

class RecordNotFoundError(DatabaseError):
    """Raised when a requested record is absent."""


In [ ]:
# Step 2: Catching specific exceptions vs general
def find_user(user_id: int):
    if user_id <= 0:
        raise RecordNotFoundError(f'User ID {user_id} does not exist')
    return {'id': user_id, 'name': 'Alice'}


In [ ]:
# Step 3: Handling domain exceptions cleanly
try:
    user = find_user(-1)
except RecordNotFoundError as exc:
    print(f'Handled expected error: {exc}')


### 🔮 Prediction Prompt
**Before running the next cell:** When you catch a low-level error (like `sqlite3.OperationalError`) and raise a high-level `DatabaseError`, what is the difference between `raise DatabaseError('failed')` vs `raise DatabaseError('failed') from original_error` in the traceback?


In [ ]:
# Surprising Result: Explicit Exception Chaining (__cause__ vs __context__)

try:
    try:
        _ = 1 / 0
    except ZeroDivisionError as low_level:
        raise DatabaseError('Transaction failed') from low_level
except DatabaseError as high_level:
    print(f'Caught: {high_level}')
    print(f'Explicit Root Cause (__cause__): {high_level.__cause__}')
    print('Explanation: "raise from" links the low-level cause directly for APM monitors!')


### Structured JSON Logging
In microservices, text logs are difficult to index. Structured JSON logs allow tools like Datadog and Elasticsearch to filter by user, latency, and status code.


In [ ]:
import datetime
import json


def log_event(level: str, message: str, **context):
    entry = {
        'timestamp': datetime.datetime.now(datetime.UTC).isoformat(),
        'level': level.upper(),
        'message': message,
        **context
    }
    print(json.dumps(entry))

log_event('info', 'Payment processed', user_id=42, amount_cents=1999, currency='USD')


### Python 3.11+ Exception Groups (`ExceptionGroup`)
When running concurrent tasks or validation pipelines, multiple independent errors can occur simultaneously.


In [ ]:
eg = ExceptionGroup(
    'Batch validation failure',
    [ValueError('Invalid age: -5'), TypeError('Expected string for email'), KeyError('Missing zip')]
)
print(f'ExceptionGroup contains {len(eg.exceptions)} separate errors:')
for exc in eg.exceptions:
    print(f'  - {type(exc).__name__}: {exc}')


### 🛠️ Interactive Challenge: Fix the Silent Error Swallowing
The following function uses a bare `except:` or catch-all block that silently swallows a fatal bug (like `KeyboardInterrupt` or a typo in a variable name). Fix it to catch only domain-specific exceptions.


In [ ]:
# TODO: FIX ME - Narrow the exception catch block to only handle ValueError
# Currently it catches everything, hiding the NameError typo bug!

def process_order(price_str: str):
    try:
        price = float(price_str)
        # Bug: undefined variable total_wth_tax
        result = total_with_tax * price
        return result
    except ValueError as e:
        # FIX: Catch ValueError only, so NameError is surfaced!
        print(f'Invalid price format: {e}')
        return 0.0
    except NameError:
        print('Caught and detected the developer typo bug!')
        return 0.0

process_order('19.99')


### 🏁 Summary & Next Steps
- Always derive custom exceptions from a base `ApplicationError`.
- Use `raise NewError() from orig_error` to preserve root causes.
- Run `python 01_exceptions_deep_dive_demo.py` and `python 02_structured_logging_demo.py`.
- Follow [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to implement the structured logger.
